# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# ML-09 — Validation and Research Claim Audit

## 1. Two paper findings + my methodology questions

### Finding 1
**Finding:** [Paper finding]

**Methodology question:**  
How was the label/outcome used for this finding defined and obtained? I would want to understand whether the label was available independently of the model inputs and whether the labeling process could introduce bias.

**Validation question:**  
Does the validation design match the claim being made? In particular, I would want to know whether the evaluation split prevents related observations from appearing across train and validation sets and whether the validation setting reflects the intended real-world use.

### Finding 2
**Finding:** [Paper finding]

**Methodology question:**  
How was the outcome measured, and does that measurement directly represent the claim being made?

**Validation question:**  
Does the evaluation design provide enough evidence to generalize this result beyond the evaluated sample?

### Review stance

These questions are intended as constructive methodology checks rather than criticisms of the research. The same questions should also be applied to my own model.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [12]:
!git clone https://github.com/Abhaykr45/flyrank-ml-internship.git

import pandas as pd

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 128 (delta 42), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.89 MiB | 10.52 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [14]:
print(df.columns.tolist())


['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', '

In [17]:
# Look for possible client/group columns
possible_group_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in ["client", "group", "site", "domain"])
]

print("Possible group columns:")
print(possible_group_cols)
# Look for possible date/time columns
possible_time_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in ["date", "time", "month", "year"])
]

print("Possible time columns:")
print(possible_time_cols)

Possible group columns:
['client_id']
Possible time columns:
['days_since_last_update']


In [19]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

In [20]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "trend_direction"

data = df[features + [target, "client_id"]].dropna()

X = data[features]
y = data[target]
groups = data["client_id"]

print("Rows:", len(data))
print("Unique clients:", groups.nunique())

Rows: 20018
Unique clients: 29


In [21]:
from sklearn.model_selection import train_test_split

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_model.fit(X_train_random, y_train_random)

random_predictions = random_model.predict(X_test_random)

random_accuracy = accuracy_score(
    y_test_random,
    random_predictions
)

print("Week-5 Random Split Accuracy:", random_accuracy)

Week-5 Random Split Accuracy: 0.6323676323676324


In [22]:
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

In [23]:
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(overlap))

Training clients: 23
Testing clients: 6
Client overlap: 0


In [24]:
group_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

group_model.fit(
    X_train_group,
    y_train_group
)

group_predictions = group_model.predict(X_test_group)

group_accuracy = accuracy_score(
    y_test_group,
    group_predictions
)

print("Week-6 Grouped Split Accuracy:", group_accuracy)

Week-6 Grouped Split Accuracy: 0.5839681440443213


In [25]:
comparison = pd.DataFrame({
    "Validation": [
        "Week-5 Random Split",
        "Week-6 Client-Grouped Split"
    ],
    "Accuracy": [
        random_accuracy,
        group_accuracy
    ]
})

comparison

,Validation,Accuracy
0,Week-5 Random Split,0.632368
1,Week-6 Client-Grouped Split,0.583968


### Before vs. After Interpretation

The Random Forest model showed an observed accuracy of 0.632368 under the original Week-5 random split.

Under the client-grouped split, the measured accuracy was 0.583968.

The grouped result was 4.84 percentage points lower than the original random-split result. This suggests that the random split may have provided a more optimistic estimate of performance because observations from the same client could be present in both the training and testing sets.

The client-grouped split provides a more demanding evaluation of performance on unseen clients. Based on this audit, I would describe the model as providing directional decision-support rather than claiming that it will generalize equally well to new clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audited the final feature set used by my Week-5 Random Forest model for possible target leakage.

The target is `trend_direction`. A feature would be considered leakage if it contains information that would only be known after, or as a direct consequence of, the target outcome.

The audit focuses on whether each feature would be available at the time a content-refresh decision is made.

In [26]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "trend_direction"

print("Target:", target)
print("\nFeatures being audited:")
for feature in features:
    print("-", feature)

Target: trend_direction

Features being audited:
- search_volume
- competition
- cpc
- word_count
- char_count
- content_age_days
- days_since_last_update
- ctr
- avg_position


In [27]:
# Look for columns whose names may directly describe the target or future outcome

suspicious_columns = [
    col for col in df.columns
    if any(keyword in col.lower()
           for keyword in ["trend", "target", "label", "outcome", "future", "next"])
]

print("Potentially suspicious columns:")
print(suspicious_columns)

Potentially suspicious columns:
['trend_direction', 'trend_pct']


In [28]:
# Check whether any feature has exactly the same values as the target

for feature in features:
    if df[feature].equals(df[target]):
        print(f"WARNING: {feature} exactly matches {target}")
    else:
        print(f"{feature}: no exact match")

search_volume: no exact match
competition: no exact match
cpc: no exact match
word_count: no exact match
char_count: no exact match
content_age_days: no exact match
days_since_last_update: no exact match
ctr: no exact match
avg_position: no exact match


In [29]:
leakage_audit = pd.DataFrame({
    "Feature": features,
    "Leakage_Risk": [
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Potential",
        "Potential"
    ],
    "Reason": [
        "Search demand can be observed independently of the target.",
        "Competition is an input signal and does not directly encode the target.",
        "CPC is an observed market signal and does not directly encode the target.",
        "Word count is a property of the existing content.",
        "Character count is a property of the existing content.",
        "Content age can be known when making the decision.",
        "Days since last update can be known when making the decision.",
        "CTR is observed performance data, but its timing relative to the target should be verified.",
        "Average position is observed performance data, but its timing relative to the target should be verified."
    ]
})

leakage_audit

,Feature,Leakage_Risk,Reason
0,search_volume,Low,Search demand can be observed independently of...
1,competition,Low,Competition is an input signal and does not di...
2,cpc,Low,CPC is an observed market signal and does not ...
3,word_count,Low,Word count is a property of the existing content.
4,char_count,Low,Character count is a property of the existing ...
5,content_age_days,Low,Content age can be known when making the decis...
6,days_since_last_update,Low,Days since last update can be known when makin...
7,ctr,Potential,"CTR is observed performance data, but its timi..."
8,avg_position,Potential,"Average position is observed performance data,..."


### Leakage audit interpretation

The audit did not identify an obvious direct copy of the target in the final feature set.

Most features describe observable content properties or performance signals that could reasonably be available when making a content-refresh decision.

However, `ctr` and `avg_position` require particular attention because their validity depends on when they were measured relative to the `trend_direction` label. If these measurements were collected after the target period, they could introduce information that would not have been available at prediction time.

Therefore, I treat these features as potential timing risks until their measurement window is confirmed. The audit supports a cautious interpretation of the model rather than assuming that every feature is automatically leakage-free.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite
### Original claim

The Random Forest model achieved higher accuracy than the Week 4 baseline, indicating that it captures more complex relationships in the data and can help identify content that may require refreshing.
### Rewritten claim

The Random Forest model showed higher observed accuracy than the Week 4 baseline under the original random-split evaluation. Its measured accuracy was 0.632368 under that split and 0.583968 under the client-grouped split.

The lower accuracy on unseen clients suggests that performance may not generalize equally across clients. Therefore, I would describe the model as providing directional decision-support for content-refresh prioritization rather than as an automatic or guaranteed predictor of which content should be refreshed.

The results are observed and measured on the evaluated dataset and validation designs, and further validation on new data would be needed before making stronger generalization claims.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.